# OCRia-1 manifest runner (Colab A100)

Runs a docparse manifest of OCRia-1 page calls with vLLM. Run only after the owner has seen `python -m docparse cache plan` for the manifest and agreed.

1. Upload the manifest folder (`manifest.jsonl` + `blobs/`) to `MyDrive/docparse/manifests/<name>`.
2. Runtime → Change runtime type → A100.
3. Run all cells. Answers are appended to `<name>/results.jsonl` on Drive; a disconnected session resumes where it stopped.

In [ ]:
MANIFEST = 'pilot'  # folder under MyDrive/docparse/manifests
VLLM_VERSION = '0.34.0'  # OCRia-1 needs 0.30 or later; must match the runtime string of the calls
OCRIA_REVISION = 'dae83aa51ecd0a8574d415d929a4d9e79348fd2a'
WORKERS = 32

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "vllm=={VLLM_VERSION}" httpx pydantic zstandard pillow pypdfium2 markdown2 pyyaml
!rm -rf /content/parse-bench-2 && git clone -q https://github.com/RohithReddy20/parse-bench-2 /content/parse-bench-2

In [ ]:
import subprocess, time, httpx
proc = subprocess.Popen(['vllm', 'serve', 'llamaindex/OCRia-1', '--revision', OCRIA_REVISION, '--served-model-name', 'ocria-1',
                         '--dtype', 'bfloat16', '--max-model-len', '32768', '--limit-mm-per-prompt', '{"image": 1}', '--port', '8000'])
while True:
    assert proc.poll() is None, 'vllm exited'
    try:
        if httpx.get('http://localhost:8000/health', timeout=5).status_code == 200: break
    except httpx.HTTPError: pass
    time.sleep(5)
print('vllm', httpx.get('http://localhost:8000/version').json())

In [ ]:
import sys
sys.path[:0] = ['/content/parse-bench-2/src', '/content/parse-bench-2']
from pathlib import Path
from huggingface_hub import hf_hub_download
from docparse.models.manifest import read_manifest
from serve.executors import openai_chat_executor
from serve.run_manifest import run_manifest

mdir = Path('/content/drive/MyDrive/docparse/manifests') / MANIFEST
assert all(c.runtime == f'vllm-{VLLM_VERSION}' and c.model.revision == OCRIA_REVISION for c, _ in read_manifest(mdir)), 'manifest built for another runtime or revision'
prompt = Path(hf_hub_download('llamaindex/OCRia-1', 'prompt.txt', revision=OCRIA_REVISION)).read_text()
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
executor = openai_chat_executor('http://localhost:8000/v1', 'ocria-1', prompt, gpu_type=gpu)
print(run_manifest(mdir, mdir / 'results.jsonl', executor, max_workers=WORKERS))

In [ ]:
proc.terminate()
from google.colab import runtime
runtime.unassign()  # stop spending compute units